# TFC - iEquus - Raio-x

## Livrarias

In [34]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights, vit_b_32, ViT_B_32_Weights, mobilenet_v3_small, MobileNet_V3_Small_Weights, vgg11, VGG11_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score ,confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import PIL
from PIL import Image
import time

## Settings

In [35]:
num_classes = 5  # daisy, dandelion, rose, sunflower, tulip
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed = 42
torch.manual_seed(seed)

## Data loader

In [36]:
def diminuir_data():
    original = "flowers"
    base = "flower_500"

    #Este codigo cria 1 pasta com 2 diretórios que vai ser guardado as img de teste e de treino
    for split in ["train", "test"]:
        for cls in os.listdir(original):
            os.makedirs(os.path.join(base, split, cls), exist_ok=True)


    for classe in os.listdir(original): #Vai em pasta em pasta dentro da pasta "flowers" - classe = nome da flor

        train_dir = os.path.join(base, "train", classe)
        test_dir = os.path.join(base, "test", classe)

        classe_path = os.path.join(original, classe) #Junta o caminho base com o nome da classe.
        imagens = os.listdir(classe_path)[:100] #Lista os nomes das 100º as imagens dentro dessa pasta.

        imagens = [os.path.join(classe_path, img) for img in imagens] #Lista com todas as imagens associada a sua classe

        #Dividir o dataset em treino e teste
        treino, teste = train_test_split(imagens, test_size = 0.2, random_state=42)

        for img_path in treino:
            dest = os.path.join(base, "train", classe, os.path.basename(img_path))
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy(img_path, dest)
            
        for img_path in teste:
            dest = os.path.join(base, "test", classe, os.path.basename(img_path))
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy(img_path, dest)

In [37]:
training_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=(224, 224)),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomRotation(degrees=30, interpolation=PIL.Image.BILINEAR),
    torchvision.transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # tamanho padrão da ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset_500 = datasets.ImageFolder('flower_500/train', transform=training_transforms)
test_dataset_500 = datasets.ImageFolder('flower_500/test', transform=test_transforms)

train_loader_500 = DataLoader(train_dataset_500, batch_size=64, shuffle=True) #diminui o tamanho das batches
test_loader_500 = DataLoader(test_dataset_500, batch_size=64, shuffle=False)


## Funções:

In [38]:
def best_epoch(nome, model, num_epochs=150):
    best_f1 = 0
    best_epoch = 0

    if hasattr(model, "fc"):
        optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
    elif hasattr(model, "heads"):
        optimizer = torch.optim.Adam(model.heads.head.parameters(), lr=0.001)
    elif hasattr(model, "classifier"):
        optimizer = torch.optim.Adam(model.classifier[3].parameters(), lr=0.001)
    
    criterion = nn.CrossEntropyLoss()

    #Utilizar CPU ou cuda
    model.to(device)

    #Treinar o modelo
    for epoch in range(1, num_epochs +1):
        model.train()
        running_loss = 0.0

        for imagens, labels in train_loader_500:
            imagens = imagens.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(imagens)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        model.eval()
        previsoes = []
        real = []

        with torch.no_grad():
            for imagens, labels in test_loader_500:
                imagens, labels = imagens.to(device), labels.to(device)
                outputs = model(imagens) #outputs é um tensor que contem as probabilidades da imagem ser de cada label
                
                _, predicted = torch.max(outputs.data, 1) #Das probabilidades pega a maior e assume ela como o label
                
                previsoes.extend(predicted.cpu().numpy()) #temos que passar para numpy pois é um tensor, e o sklearn n trabalha com isso
                real.extend(labels.cpu().numpy())
        
        #calcular metricas
        f1 = f1_score(real, previsoes, average="macro") 

        if f1 > best_f1:
            best_f1 = f1
            best_epoch = epoch
            best_prev = previsoes
            best_real = real

            torch.save(model.state_dict(), f"model_{nome}.pth")
    
    print(f"O melhor f1 foi {best_f1}, na epoch num {best_epoch}")

    return best_prev, best_real

In [46]:
def metricas(nome, real, previsoes):
    accuracy = accuracy_score(real, previsoes)
    precision = precision_score(real, previsoes, average="macro") #average=None - Mostra a precision para cada classe
    recall = recall_score(real, previsoes, average="macro") #average=None - Mostra a recall para cada classe
    f1 = f1_score(real, previsoes, average="macro") #average=None - Mostra o f1 para cada classe

    print(f"""Accuracy: {accuracy},\nPrecision: {precision},\nRecall: {recall},\nF1: {f1}""")
    
    test_data = datasets.ImageFolder('flower_split/test', transform = test_transforms)
    classes = test_data.classes
    matriz = confusion_matrix(real, previsoes)
    display = ConfusionMatrixDisplay(matriz, display_labels=classes)
    display.plot()
    display.figure_.suptitle("Matriz de confusão")
    plt.savefig(f"{nome}_confusion_matrix", bbox_inches='tight')  
    plt.close()

    with open("metricas.txt", "a") as f:
        f.write(f"""{nome}\nAccuracy: {accuracy},\nPrecision: {precision},\nRecall: {recall},\nF1: {f1}\n\n\n""")

In [56]:
def testagem(model, imagem):
    if model == "ResNet18":
        modelo = resnet18(num_classes=5)

    elif model == "ResNet50":
        modelo = resnet50(num_classes=5)
        
    elif model == "ViT":
        modelo = vit_b_32(num_classes=5)

    else:
        modelo = mobilenet_v3_small(num_classes=5)
    
    modelo.load_state_dict(torch.load(f"model_{model}.pth"))
    modelo.eval()
    modelo.to(device)
    
    s = time.process_time()

    with torch.no_grad():
        output = modelo(imagem)
        i, predicted = torch.max(output, 1)

    e = time.process_time()
    
    print(f"Classe prevista: {predicted.item()} em {e - s} segundos")

## Modelos

In [40]:
# Modelo ResNet18
res18_model = resnet18(ResNet18_Weights.DEFAULT)
for param in res18_model.parameters():
    param.requires_grad = False

res18_model.fc = nn.Linear(res18_model.fc.in_features, num_classes)

# Modelo ResNet50
res50_model = resnet50(ResNet50_Weights.DEFAULT)
for param in res50_model.parameters():
    param.requires_grad = False

res50_model.fc = nn.Linear(res50_model.fc.in_features, num_classes)

# Modelo ViT
vit_model = vit_b_32(ViT_B_32_Weights.DEFAULT) 
for vit_param in vit_model.parameters():
    vit_param.requires_grad = False

vit_model.heads.head = nn.Linear(vit_model.heads.head.in_features, num_classes)

# Modelo mobilev3
mobile_v3 = mobilenet_v3_small(MobileNet_V3_Small_Weights.DEFAULT)
for mobilev3_param in mobile_v3.parameters():
    mobilev3_param.requires_grad = False

mobile_v3.classifier[3] = nn.Linear(mobile_v3.classifier[3].in_features, num_classes)

c:\Users\Utilizador\Documents\BACKUP MAT\Manter\Faculdade\3 ano\Trabalho final de curso\Codigos\.venv\Lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


In [41]:
res18_prev, res18_real = best_epoch("ResNet18", res18_model)

O melhor f1 foi 0.9098326045694467, na epoch num 99


In [42]:
res50_prev, res50_real = best_epoch("ResNet50", res50_model)

O melhor f1 foi 0.9397069597069597, na epoch num 93


In [43]:
vit_prev, vit_real = best_epoch("ViT", vit_model)

O melhor f1 foi 0.9403696859409312, na epoch num 109


In [44]:
mobile_v3_prev, mobile_v3_real = best_epoch("MobileNetV3", mobile_v3)

O melhor f1 foi 0.9392182614133834, na epoch num 32


In [47]:
metricas("ResNet18", res18_prev, res18_real)
metricas("ResNet50", res50_prev, res50_real)
metricas("ViT", vit_prev, vit_real)
metricas("MobileNetV3", mobile_v3_prev, mobile_v3_real)

Accuracy: 0.91,
Precision: 0.9099999999999999,
Recall: 0.9171029177221127,
F1: 0.9098326045694467
Accuracy: 0.94,
Precision: 0.9400000000000001,
Recall: 0.9407655502392345,
F1: 0.9397069597069597
Accuracy: 0.94,
Precision: 0.9400000000000001,
Recall: 0.943001443001443,
F1: 0.9403696859409312
Accuracy: 0.94,
Precision: 0.9400000000000001,
Recall: 0.93937343358396,
F1: 0.9392182614133834


In [55]:
imagem = Image.open("teste_rosa.jpg").convert("RGB")
imagem = test_transforms(imagem).unsqueeze(0)

testagem("ResNet18", imagem)
testagem("ResNet50", imagem)
testagem("ViT", imagem)
testagem("MobileNetV3", imagem)

Classe prevista: 2 em 0.375 segundos
Classe prevista: 2 em 0.75 segundos
Classe prevista: 2 em 0.65625 segundos
Classe prevista: 2 em 0.25 segundos
